# 95 — Agent Safety Sandbox

## Goal
Build an agent with **approval gates**, **tool allowlists**,
and **refusal behavior** for risky operations — essential
safety controls for production agentic systems.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Approval gates** | Human-in-the-loop for dangerous actions |
| **Tool allowlists** | Only permitted tools can execute |
| **Risk classification** | Auto-classify tool calls as safe/risky |
| **Refusal behavior** | Gracefully decline unsafe requests |

## Stack
- `langgraph` — StateGraph with safety gate nodes
- `langchain-ollama` — local LLM
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
from datetime import datetime
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Define Tools with Risk Levels

Each tool has a risk classification:
- 🟢 **SAFE** — read-only, no side effects
- 🟡 **MODERATE** — writes data but reversible
- 🔴 **DANGEROUS** — destructive, financial, or irreversible

In [ ]:
TOOLS = {
    "read_file": {
        "risk": "SAFE", "description": "Read contents of a file",
        "fn": lambda path: f"Contents of {path}: [sample data...]"
    },
    "list_directory": {
        "risk": "SAFE", "description": "List files in a directory",
        "fn": lambda path: f"Files in {path}: file1.py, file2.py, config.yaml"
    },
    "search_logs": {
        "risk": "SAFE", "description": "Search application logs",
        "fn": lambda query: f"Log results for '{query}': 42 entries found"
    },
    "write_file": {
        "risk": "MODERATE", "description": "Write or update a file",
        "fn": lambda path: f"File written: {path}"
    },
    "restart_service": {
        "risk": "MODERATE", "description": "Restart an application service",
        "fn": lambda svc: f"Service {svc} restarted"
    },
    "delete_file": {
        "risk": "DANGEROUS", "description": "Permanently delete a file",
        "fn": lambda path: f"DELETED: {path}"
    },
    "drop_database_table": {
        "risk": "DANGEROUS", "description": "Drop a database table permanently",
        "fn": lambda table: f"TABLE DROPPED: {table}"
    },
    "transfer_funds": {
        "risk": "DANGEROUS", "description": "Transfer money between accounts",
        "fn": lambda details: f"TRANSFER: {details}"
    },
    "execute_shell": {
        "risk": "DANGEROUS", "description": "Execute a shell command on the server",
        "fn": lambda cmd: f"EXECUTED: {cmd}"
    },
}

# Allowlist: tools the agent is permitted to use at all
TOOL_ALLOWLIST = {"read_file", "list_directory", "search_logs", "write_file",
                  "restart_service", "delete_file"}
# Blocked even if requested:
TOOL_BLOCKLIST = {"execute_shell", "drop_database_table", "transfer_funds"}

print("Tool registry:")
for name, t in TOOLS.items():
    status = "✅ allowed" if name in TOOL_ALLOWLIST else "🚫 blocked"
    risk_icon = {"SAFE": "🟢", "MODERATE": "🟡", "DANGEROUS": "🔴"}[t['risk']]
    print(f"  {risk_icon} {name:25s} [{t['risk']:10s}] {status}")

## Step 2 — Safety Agent Graph

```
parse_request → classify_risk → [risk level?]
    → BLOCKED:    refuse → END
    → SAFE:       execute → respond → END
    → MODERATE:   execute → respond → END
    → DANGEROUS:  approval_gate → [approved?]
                      → YES: execute → respond → END
                      → NO:  refuse → END
```

In [ ]:
class SafetyState(TypedDict):
    user_request: str
    parsed_tool: str
    parsed_args: str
    risk_level: str
    blocked: bool
    approval_status: str  # "pending", "approved", "denied"
    execution_result: str
    response: str
    audit_log: list

print("State schema defined")

In [ ]:
def parse_request(state: SafetyState) -> SafetyState:
    """LLM identifies which tool the user wants."""
    print(f"\n📨 Request: {state['user_request']}")
    
    tools_list = "\n".join(f"- {name}: {t['description']}" for name, t in TOOLS.items())
    msg = llm.invoke([
        SystemMessage(content=f"""Identify which tool the user wants to use.
Available tools:
{tools_list}

Return JSON: {{"tool": "tool_name", "args": "argument string"}}
If no tool matches, return: {{"tool": "none", "args": ""}}
Return ONLY the JSON."""),
        HumanMessage(content=state["user_request"])
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    
    try:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
        tool_name = parsed.get("tool", "none")
        args = parsed.get("args", "")
    except (json.JSONDecodeError, ValueError):
        tool_name = "none"
        args = ""
    
    print(f"   Parsed tool: {tool_name}, args: {args}")
    return {**state, "parsed_tool": tool_name, "parsed_args": str(args)}

print("Node: parse_request defined")

In [ ]:
def classify_risk(state: SafetyState) -> SafetyState:
    """Check allowlist and classify risk."""
    tool_name = state["parsed_tool"]
    audit = list(state.get("audit_log", []))
    
    # Check blocklist first
    if tool_name in TOOL_BLOCKLIST:
        print(f"   🚫 BLOCKED: '{tool_name}' is on the blocklist")
        audit.append({"action": "BLOCKED", "tool": tool_name,
                      "time": datetime.now().isoformat(), "reason": "blocklist"})
        return {**state, "risk_level": "BLOCKED", "blocked": True, "audit_log": audit}
    
    # Check allowlist
    if tool_name not in TOOL_ALLOWLIST:
        print(f"   🚫 NOT ALLOWED: '{tool_name}' not in allowlist")
        audit.append({"action": "BLOCKED", "tool": tool_name,
                      "time": datetime.now().isoformat(), "reason": "not in allowlist"})
        return {**state, "risk_level": "BLOCKED", "blocked": True, "audit_log": audit}
    
    # Get risk level from tool definition
    risk = TOOLS.get(tool_name, {}).get("risk", "DANGEROUS")
    icon = {"SAFE": "🟢", "MODERATE": "🟡", "DANGEROUS": "🔴"}[risk]
    print(f"   {icon} Risk level: {risk}")
    
    audit.append({"action": "CLASSIFIED", "tool": tool_name,
                  "risk": risk, "time": datetime.now().isoformat()})
    return {**state, "risk_level": risk, "blocked": False, "audit_log": audit}

def risk_router(state: SafetyState) -> str:
    if state["blocked"]:
        return "refuse"
    if state["risk_level"] == "DANGEROUS":
        return "approval_gate"
    return "execute"  # SAFE or MODERATE

print("Node: classify_risk defined")

In [ ]:
def approval_gate(state: SafetyState) -> SafetyState:
    """Simulate human approval for dangerous actions."""
    print(f"   ⏸️ APPROVAL REQUIRED for: {state['parsed_tool']}({state['parsed_args']})")
    
    # In production: wait for human approval via UI/Slack/email
    # For demo: auto-approve delete_file, deny others
    if state["parsed_tool"] == "delete_file":
        print("   ✅ Simulated: APPROVED (delete_file with backup)")
        status = "approved"
    else:
        print("   ❌ Simulated: DENIED (too risky)")
        status = "denied"
    
    audit = list(state["audit_log"]) + [
        {"action": f"APPROVAL_{status.upper()}", "tool": state["parsed_tool"],
         "time": datetime.now().isoformat()}
    ]
    return {**state, "approval_status": status, "audit_log": audit}

def approval_router(state: SafetyState) -> str:
    return "execute" if state["approval_status"] == "approved" else "refuse"

def execute_tool_node(state: SafetyState) -> SafetyState:
    """Execute the approved tool."""
    tool_name = state["parsed_tool"]
    args = state["parsed_args"]
    print(f"   ⚙️ Executing: {tool_name}({args})")
    
    fn = TOOLS[tool_name]["fn"]
    result = fn(args)
    print(f"   Result: {result}")
    
    audit = list(state["audit_log"]) + [
        {"action": "EXECUTED", "tool": tool_name, "args": args,
         "time": datetime.now().isoformat()}
    ]
    return {**state, "execution_result": result, "response": result, "audit_log": audit}

def refuse_node(state: SafetyState) -> SafetyState:
    """Refuse the request gracefully."""
    tool_name = state["parsed_tool"]
    if state["blocked"]:
        msg = (f"I'm sorry, I cannot execute '{tool_name}'. "
               f"This tool is not permitted by the safety policy.")
    else:
        msg = (f"The action '{tool_name}' requires human approval and was denied. "
               f"Please contact your admin for manual execution.")
    
    print(f"   🚫 Refused: {msg}")
    audit = list(state["audit_log"]) + [
        {"action": "REFUSED", "tool": tool_name, "time": datetime.now().isoformat()}
    ]
    return {**state, "response": msg, "audit_log": audit}

print("Nodes: approval_gate, execute, refuse defined")

In [ ]:
# Build the safety-sandboxed graph
graph = StateGraph(SafetyState)

graph.add_node("parse", parse_request)
graph.add_node("classify", classify_risk)
graph.add_node("approval_gate", approval_gate)
graph.add_node("execute", execute_tool_node)
graph.add_node("refuse", refuse_node)

graph.set_entry_point("parse")
graph.add_edge("parse", "classify")
graph.add_conditional_edges("classify", risk_router, {
    "execute": "execute",
    "approval_gate": "approval_gate",
    "refuse": "refuse"
})
graph.add_conditional_edges("approval_gate", approval_router, {
    "execute": "execute",
    "refuse": "refuse"
})
graph.add_edge("execute", END)
graph.add_edge("refuse", END)

safety_app = graph.compile()
print("Safety sandbox graph compiled!")

In [ ]:
# Test with different risk levels
test_requests = [
    "Read the file /etc/config.yaml",                    # SAFE
    "Write 'hello' to output.txt",                        # MODERATE
    "Delete the file /tmp/old_backup.tar.gz",              # DANGEROUS (allowed, needs approval)
    "Execute the command 'rm -rf /' on the server",        # BLOCKED
    "Transfer $50,000 to account 12345",                   # BLOCKED
    "Search logs for 'error 500' in the last hour",        # SAFE
]

all_results = []
for req in test_requests:
    print("\n" + "=" * 60)
    result = safety_app.invoke({
        "user_request": req,
        "parsed_tool": "",
        "parsed_args": "",
        "risk_level": "",
        "blocked": False,
        "approval_status": "pending",
        "execution_result": "",
        "response": "",
        "audit_log": []
    })
    all_results.append(result)

# Summary
print("\n\n" + "=" * 60)
print("SAFETY DASHBOARD")
print("=" * 60)
for r in all_results:
    risk_icon = {"SAFE": "🟢", "MODERATE": "🟡", "DANGEROUS": "🔴", "BLOCKED": "🚫"}.get(r['risk_level'], '⚪')
    print(f"  {risk_icon} [{r['risk_level']:10s}] {r['parsed_tool']:25s} → {r['response'][:60]}")

In [ ]:
# Show full audit trail
print("AUDIT TRAIL")
print("=" * 60)
for i, r in enumerate(all_results):
    print(f"\nRequest {i+1}: {r['user_request'][:50]}")
    for entry in r["audit_log"]:
        print(f"   [{entry['time'][:19]}] {entry['action']} — {entry.get('tool', '')} {entry.get('reason', '')}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Allowlist** | Only permitted tools can be invoked |
| **Blocklist** | Certain tools are always refused |
| **Risk classification** | SAFE / MODERATE / DANGEROUS per tool |
| **Approval gate** | Dangerous actions require human approval |
| **Audit trail** | Every action is logged with timestamps |
| **Graceful refusal** | Denied requests get helpful error messages |

## Safety Architecture
```
User Request → Parse → Classify Risk
                        ├─ BLOCKED → Refuse
                        ├─ SAFE → Execute
                        ├─ MODERATE → Execute
                        └─ DANGEROUS → Approval Gate
                                       ├─ Approved → Execute
                                       └─ Denied → Refuse
```

## Extending This Project
- Real approval via Slack webhook or email
- Rate limiting (max N dangerous calls per hour)
- Role-based access (admin vs user vs viewer)
- Audit log persistence to database